# 各プレイヤーのvoronoi areaの相互相関係数の計算

## パス関係の定義

In [1]:
import sys
from pathlib import Path

# src ディレクトリのパスを取得して追加
src_dir = str(Path().resolve().parents[1])
if src_dir not in sys.path:
    sys.path.append(src_dir)

## プレイヤーの組み合わせの決定

In [2]:
import itertools

from analyzers import datamanager

data_manager = datamanager.DataManager("../processed_data")

all_player_names = data_manager.get_all_player_names()
teams = set([p[0] for p in all_player_names])

combinations = []

for team in teams:
    team_combinations = itertools.combinations(
        [p for p in all_player_names if p[0] == team], 2
    )

    print(f"Team {team} combinations:")
    for comb in team_combinations:
        print(f"  {comb}")
        combinations.append(comb)

print(f"All Combinations: {combinations}")

Match Type:  odict_keys(['AB', 'BD', 'CB', 'G1', 'G2', 'G3', 'G4', 'after', 'before'])
Team D combinations:
  ('D1black', 'D2orange')
  ('D1black', 'D3yellow')
  ('D2orange', 'D3yellow')
Team O combinations:
  ('O1red', 'O2blue')
  ('O1red', 'O3pink')
  ('O2blue', 'O3pink')
All Combinations: [('D1black', 'D2orange'), ('D1black', 'D3yellow'), ('D2orange', 'D3yellow'), ('O1red', 'O2blue'), ('O1red', 'O3pink'), ('O2blue', 'O3pink')]


## 描画

In [3]:
import math
import matplotlib.pyplot as plt

from analyzers import calculator
from analyzers.cross_correlation import drawer

all_trajectories = data_manager.get_all_trajectories()

for comb in combinations:
    player_one_data = data_manager.get_data_by_player(comb[0])
    player_two_data = data_manager.get_data_by_player(comb[1])
    
    for match_type in player_one_data.keys():
        player_one_games = player_one_data[match_type]
        player_two_games = player_two_data[match_type]
        
        size_of_ax = (15, 7)
        num_axes = len(player_one_games)
        if num_axes > 2:
            num_cols = 3
        else:
            num_cols = num_axes
        num_rows = math.ceil(num_axes / num_cols)
        fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(size_of_ax[0]*num_cols, size_of_ax[1]*num_rows))

        axes = axes.flatten()

        for idx, game in enumerate(player_one_games.keys()):
            player_one_trajectory = calculator.z_normarize(player_one_games[game])
            player_two_trajectory = calculator.z_normarize(player_two_games[game])
            
            drawer.plot_cross_correlation(
                data=[player_one_trajectory, player_two_trajectory],
                ax=axes[idx],
                normed=True,
                linefmt='grey',
                markerfmt=',',
                basefmt='black'
            )
        
            axes[idx].set_title(f'{comb[0]}-{comb[1]} in {game}')

        fig.suptitle(f"Cross correlation in match {match_type}")
        
        # 画像の保存処理    
        results_dir = Path(f'./results/{match_type}')
        results_dir.mkdir(parents=True, exist_ok=True)
        plt.savefig(f'./results/{match_type}/{comb[0]}-{comb[1]}', bbox_inches='tight')
        # plt.show()
        plt.close()    